# GuardBench downloaded dataset stats

Scans the on-disk caches under **`GUARDBENCH_BASE_PATH/datasets`** (the path returned by `guardbench.paths.datasets_path()`). For each formatted split (`*.jsonl`):

- **Unsafe** — rows where **`label`** is `True` (GuardBench convention: harmful / policy-violating prompt).
- **Safe** — rows where **`label`** is `False`.
- **Categories** — count of distinct non-empty per-row **`category`** values when that field appears in the split (many datasets omit `category`; then **distinct_categories** shows as missing).
- **Taxonomy (metadata)** — number of strings in **`hazard_categories`** on the matching GuardBench dataset class when the folder name maps to a known alias (hyphens ↔ underscores).

In [1]:
from __future__ import annotations

import json

import pandas as pd

# Default ``~/.guardbench`` if unset (see ``guardbench.__init__``)
import guardbench  # noqa: F401

from guardbench.datasets import DATASETS, get_dataset
from guardbench.paths import datasets_path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

DATASETS_ROOT = datasets_path()
print("DATASETS_ROOT =", DATASETS_ROOT)

DATASETS_ROOT = /home/ahoai/.guardbench/datasets


In [2]:
from pathlib import Path

def folder_to_alias(folder_name: str) -> str | None:
    alias = folder_name.replace("-", "_")
    return alias if alias in DATASETS else None


def hazard_taxonomy_length(alias: str | None) -> int | None:
    if alias is None:
        return None
    try:
        ds = get_dataset(alias)
        hc = getattr(ds, "hazard_categories", None)
        if not hc:
            return None
        return len(hc)
    except Exception:
        return None


def analyze_jsonl(split_path: Path) -> dict[str, object]:
    """Counts safe/unsafe and distinct ``category`` values for one JSONL split."""
    safe = unsafe = 0
    categories: set[str] = set()
    rows_with_category_key = 0
    nonempty_category_rows = 0

    with split_path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            lbl = row["label"]
            if bool(lbl):
                unsafe += 1
            else:
                safe += 1

            if "category" in row:
                rows_with_category_key += 1
                cat = row.get("category")
                if cat is not None and str(cat).strip() != "":
                    nonempty_category_rows += 1
                    categories.add(str(cat))

    return {
        "safe": safe,
        "unsafe": unsafe,
        "total": safe + unsafe,
        "rows_with_category_field": rows_with_category_key,
        "distinct_categories": len(categories) if rows_with_category_key else pd.NA,
        "nonempty_category_rows": nonempty_category_rows if rows_with_category_key else pd.NA,
    }


rows: list[dict[str, object]] = []
for d in sorted(DATASETS_ROOT.iterdir()):
    if not d.is_dir():
        continue
    alias = folder_to_alias(d.name)
    tax_n = hazard_taxonomy_length(alias)
    for split_path in sorted(d.glob("*.jsonl")):
        stem = split_path.stem
        stats = analyze_jsonl(split_path)
        rows.append(
            {
                "folder": d.name,
                "alias": alias,
                "split": stem,
                **stats,
                "metadata_hazard_categories_n": tax_n,
            }
        )

summary = pd.DataFrame(rows)
summary

/home/ahoai/GuardBench/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,folder,alias,split,safe,unsafe,total,rows_with_category_field,distinct_categories,nonempty_category_rows,metadata_hazard_categories_n
0,advbench-behaviors,advbench_behaviors,test,0,520,520,0,<NA>,<NA>,8
1,advbench-strings,advbench_strings,test,0,574,574,0,<NA>,<NA>,8
2,do-anything-now-questions,do_anything_now_questions,test,0,390,390,0,<NA>,<NA>,12
3,harmbench-behaviors,harmbench_behaviors,test,0,320,320,320,7,320,7
4,harmbench-behaviors,harmbench_behaviors,val,0,80,80,80,7,80,7
5,jbb-behaviors,jbb_behaviors,test,100,100,200,200,10,200,10
6,niche-hazard-qa,niche_hazard_qa,test,0,388,388,388,6,388,6
7,strong-reject-instructions,strong_reject_instructions,test,0,213,213,213,6,213,6
8,tech-hazard-qa,tech_hazard_qa,test,0,7745,7745,7745,7,7745,7
9,xstest,xstest,test,250,200,450,0,<NA>,<NA>,1


In [3]:
# Aggregate across every row in every downloaded split
if summary.empty:
    print("No ``*.jsonl`` found under DATASETS_ROOT. Download splits first ``guardbench.datasets.download(...)``")
else:
    totals = pd.Series(
        {
            "datasets_with_any_split": summary["folder"].nunique(),
            "jsonl_splits": len(summary),
            "rows_total": int(summary["total"].sum()),
            "safe_total": int(summary["safe"].sum()),
            "unsafe_total": int(summary["unsafe"].sum()),
        }
    )
    display(totals.to_frame("value"))

,value
datasets_with_any_split,9
jsonl_splits,10
rows_total,10880
safe_total,350
unsafe_total,10530
